<a href="https://colab.research.google.com/github/thasnissam/NorthStar_Data_Analytics_Project/blob/main/Notebooks/SQL_in_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### SQL in R Analytics

####Environment setup- Installing packages and loading datasets from GitHub

In [1]:
# Load necessary libraries
install.packages("sqldf")
library(sqldf)
library(ggplot2)

# Define GitHub Raw path
base_url <- "https://raw.githubusercontent.com/thasnissam/NorthStar_Data_Analytics_Project/main/Data/"

# Import datasets
customers <- read.csv(paste0(base_url, "customers.csv"))
orders <- read.csv(paste0(base_url, "orders.csv"))
deliveries <- read.csv(paste0(base_url, "deliveries.csv"))
complaints <- read.csv(paste0(base_url, "complaints.csv"))
hubs <- read.csv(paste0(base_url, "hubs.csv"))

# Verify the data structures
str(orders)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘gsubfn’, ‘proto’, ‘RSQLite’, ‘chron’


Loading required package: gsubfn

Loading required package: proto

Warning message:
“no DISPLAY variable so Tk is not available”
Loading required package: RSQLite



'data.frame':	1250 obs. of  11 variables:
 $ order_id             : chr  "O00001" "O00002" "O00003" "O00004" ...
 $ customer_id          : chr  "C0292" "C0459" "C0161" "C0520" ...
 $ service_type         : chr  "Passenger" "Passenger" "Passenger" "Parcel" ...
 $ order_created_at     : chr  "2024-08-20 14:43:00" "2024-05-14 22:16:00" "2025-09-02 14:37:00" "2025-01-11 17:15:00" ...
 $ promised_window_hours: int  6 24 4 2 12 1 2 4 12 6 ...
 $ pickup_zone          : chr  "Airport" "North" "West" "RiverSide" ...
 $ dropoff_zone         : chr  "South" "AIRPORT" "AIRPORT" "North" ...
 $ priority_level       : chr  "Medium" "Low" "High" "Medium" ...
 $ order_value          : num  126.7 109.3 33.5 10 125.6 ...
 $ booking_channel      : chr  "App" "App" "Phone" "App" ...
 $ special_handling_flag: int  0 0 0 1 0 1 0 0 0 0 ...


####Operational Data Management (CRUD)

In [2]:
# Check the actual column names in your data frames
names(customers)
names(orders)
names(deliveries)

[1] "customer_id"          "age"                  "home_zone"           
[4] "customer_type"        "signup_date"          "loyalty_score"       
[7] "app_engagement_score" "preferred_channel"    "account_status"

[1] "order_id"              "customer_id"           "service_type"         
 [4] "order_created_at"      "promised_window_hours" "pickup_zone"          
 [7] "dropoff_zone"          "priority_level"        "order_value"          
[10] "booking_channel"       "special_handling_flag"

[1] "delivery_id"                   "order_id"                     
 [3] "driver_id"                     "vehicle_id"                   
 [5] "hub_id"                        "dispatch_time"                
 [7] "delivery_completed_at"         "delivery_status"              
 [9] "route_distance_km"             "manual_route_override_count"  
[11] "proof_of_completion_missing"   "customer_rating_post_delivery"
[13] "fuel_or_charge_cost"

CREATE: Database Extension

In [3]:
install.packages("sqldf")
library(sqldf)
library(RSQLite) # Explicitly load RSQLite for dbConnect

# Create an in-memory SQLite database connection
mydb <- dbConnect(RSQLite::SQLite(), dbname = ":memory:")

# Step 1: Create the schema extension using the connection
dbExecute(mydb, "CREATE TABLE vehicle_alerts (
    alert_id INT,
    vehicle_id TEXT,
    alert_type TEXT,
    alert_time TIMESTAMP,
    hub_id INT)")

# Step 2: Populate the table to demonstrate operational utility using the same connection
dbExecute(mydb, "INSERT INTO vehicle_alerts (alert_id, vehicle_id, alert_type, alert_time, hub_id)
       VALUES (101, 'V071', 'Engine Heat Exception', '2026-05-14 10:00:00', 8)")

# Step 3: Verify the insertion using the same connection
# Assign the result to a variable to view it, as sqldf returns a data frame
result_table <- sqldf("SELECT * FROM vehicle_alerts", connection = mydb)
print(result_table)

# Disconnect the database connection when you are done
dbDisconnect(mydb)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



[1] 0

[1] 1

  alert_id vehicle_id            alert_type          alert_time hub_id
1      101       V071 Engine Heat Exception 2026-05-14 10:00:00      8


READ (SELECT) – Investigating Operational Deviations



In [4]:
# Objective: Identify journeys with excessive manual overrides and their status
route_deviations <- sqldf("
    SELECT delivery_id, vehicle_id, manual_route_override_count, delivery_status
    FROM deliveries
    WHERE manual_route_override_count > 2
    ORDER BY manual_route_override_count DESC
")

# Print the output to verify the query results
print(head(route_deviations))

  delivery_id vehicle_id manual_route_override_count delivery_status
1     DL00473       V071                           7          OnTime
2     DL00055       V018                           5         Delayed
3     DL00085       V032                           5          OnTime
4     DL00374       V061                           5          OnTime
5     DL00672       V030                           5          OnTime
6     DL00744       V062                           5         Delayed


UPDATE – Reconciling System Discrepancies

In [5]:
# Objective: Correct status to 'Exception' for records missing completion proof
# This operation reconciles conflicts between operational logs and field reality
sqldf("UPDATE deliveries
       SET delivery_status = 'Exception'
       WHERE proof_of_completion_missing = 1 AND delivery_status = 'Completed'")

Warning message in result_fetch(res@ptr, n = n):
“`dbGetQuery()`, `dbSendQuery()` and `dbFetch()` should only be used with `SELECT` queries. Did you mean `dbExecute()`, `dbSendStatement()` or `dbGetRowsAffected()`?”


<0 x 0 matrix>

DELETE – Data Sanitization

In [6]:
# Removing inactive accounts to ensure analytical accuracy for current operations
sqldf("DELETE FROM customers WHERE account_status = 'Inactive'")

Warning message in result_fetch(res@ptr, n = n):
“`dbGetQuery()`, `dbSendQuery()` and `dbFetch()` should only be used with `SELECT` queries. Did you mean `dbExecute()`, `dbSendStatement()` or `dbGetRowsAffected()`?”


<0 x 0 matrix>

###Mathematics and Aggregate Analysis

Hub Financial Audit

In [7]:
# Objective: Calculate failure rates and energy costs per hub
hub_audit <- sqldf("
    SELECT hub_id,
           COUNT(delivery_id) AS total_jobs,
           SUM(CASE WHEN delivery_status IN ('Failed', 'Exception') THEN 1 ELSE 0 END) AS failures,
           ROUND(AVG(fuel_or_charge_cost), 2) AS avg_energy_cost
    FROM deliveries
    GROUP BY hub_id
    ORDER BY failures DESC")

print("--- Hub Performance Summary ---")
print(head(hub_audit))

[1] "--- Hub Performance Summary ---"
  hub_id total_jobs failures avg_energy_cost
1    H08        128       26           11.71
2    H05        115       23           13.69
3    H01        136       17           12.76
4    H04        127       16           13.17
5    H06        104       15           13.32
6    H07        115       14           12.92


Revenue vs. Cost Margin Analysis

In [8]:
# Objective: Join orders and deliveries to find net margin per hub
margin_analysis <- sqldf("
    SELECT d.hub_id,
           SUM(o.order_value) AS revenue,
           SUM(d.fuel_or_charge_cost) AS energy_cost,
           ROUND(SUM(o.order_value) - SUM(d.fuel_or_charge_cost), 2) AS net_margin
    FROM orders o
    JOIN deliveries d ON o.order_id = d.order_id
    GROUP BY d.hub_id
    ORDER BY net_margin ASC")

print("--- Financial Margin by Hub ---")
print(head(margin_analysis))

[1] "--- Financial Margin by Hub ---"
  hub_id  revenue energy_cost net_margin
1    H06  8977.99     1385.20    7592.79
2    H02  9577.61     1331.89    8245.72
3    H07 10342.23     1486.04    8856.19
4    H04 11438.18     1672.21    9765.97
5    H08 11356.66     1498.65    9858.01
6    H05 11472.14     1573.89    9898.25


Driver Efficiency Benchmark

In [9]:
# Objective: Calculate average rating and overrides per driver
driver_bench <- sqldf("
    SELECT driver_id,
           COUNT(delivery_id) AS volume,
           ROUND(AVG(customer_rating_post_delivery), 2) AS avg_rating,
           SUM(manual_route_override_count) AS total_overrides
    FROM deliveries
    GROUP BY driver_id
    HAVING volume > 5
    ORDER BY avg_rating ASC")

print("--- Driver Performance Benchmarking ---")
print(head(driver_bench))

[1] "--- Driver Performance Benchmarking ---"
  driver_id volume avg_rating total_overrides
1      D141      9       2.93               7
2      D165      6       2.98               8
3      D091      6       3.13               1
4      D053      7       3.14               6
5      D002      7       3.21               7
6      D016      7       3.33               7


####Executing SQL Queries for Strategic Analysis

Query 1: High-Priority Fulfillment Audit

In [10]:
# Objective: Identify high-priority orders that resulted in 'Failed' or 'Delayed' status
priority_audit <- sqldf("
    SELECT o.order_id, o.customer_id, o.priority_level, d.delivery_status, d.hub_id
    FROM orders o
    JOIN deliveries d ON o.order_id = d.order_id
    WHERE o.priority_level = 'High'
    AND d.delivery_status IN ('Failed', 'Delayed')
")
print(head(priority_audit))

  order_id customer_id priority_level delivery_status hub_id
1   O00003       C0161           High         Delayed    H02
2   O00073       C0331           High         Delayed    H05
3   O00084       C0103           High          Failed    H03
4   O00089       C0339           High          Failed    H05
5   O00113       C0640           High          Failed    H03
6   O00123       C0649           High          Failed    H08


Query 2: Peak Demand Zone Analysis

In [11]:
# Objective: Calculate total and average order value per pickup zone
zone_revenue <- sqldf("
    SELECT pickup_zone,
           COUNT(order_id) AS total_orders,
           SUM(order_value) AS total_revenue,
           ROUND(AVG(order_value), 2) AS avg_value
    FROM orders
    GROUP BY pickup_zone
    ORDER BY total_revenue DESC
    LIMIT 5
")
print(zone_revenue)

  pickup_zone total_orders total_revenue avg_value
1        East          104       9590.55     92.22
2       South          103       9517.46     92.40
3        EAST          103       9406.74     91.33
4     Airport           85       9252.42    108.85
5         Ctr           80       7559.99     94.50


Query 3: Multi-Override Exception Tracking

In [12]:
# Objective: Find special handling orders with more than 3 manual overrides
complex_exceptions <- sqldf("
    SELECT d.delivery_id, d.vehicle_id, o.service_type, d.manual_route_override_count
    FROM deliveries d
    JOIN orders o ON d.order_id = o.order_id
    WHERE o.special_handling_flag = 1
    AND d.manual_route_override_count > 3
")
print(complex_exceptions)

  delivery_id vehicle_id service_type manual_route_override_count
1     DL00193       V013    Passenger                           4
2     DL00144       V016     Business                           4
3     DL00614       V104       Parcel                           4
4     DL00935       V012     Business                           4
5     DL00881       V032    Passenger                           5
6     DL00220       V107       Retail                           4
7     DL00306       V099    Passenger                           4


Query 4: Driver Rating vs. Energy Efficiency

In [13]:
# Objective: Calculate the energy cost efficiency relative to customer satisfaction
driver_efficiency <- sqldf("
    SELECT driver_id,
           AVG(customer_rating_post_delivery) AS avg_rating,
           SUM(fuel_or_charge_cost) AS total_cost,
           ROUND(SUM(fuel_or_charge_cost) / SUM(customer_rating_post_delivery), 2) AS cost_per_rating_point
    FROM deliveries
    GROUP BY driver_id
    HAVING avg_rating > 0
    ORDER BY cost_per_rating_point ASC
")
print(head(driver_efficiency))

  driver_id avg_rating total_cost cost_per_rating_point
1      D170   3.507500      26.47                  1.89
2      D122   4.716667      27.71                  1.96
3      D012   4.380000      18.35                  2.09
4      D032   4.245000      35.70                  2.10
5      D070   4.155000      35.33                  2.13
6      D059   5.000000      10.77                  2.15


Query 5: Service-Level Profitability Audit

In [14]:
# Objective: Calculate the net margin percentage per service type
service_profitability <- sqldf("
    SELECT o.service_type,
           COUNT(o.order_id) AS total_orders,
           SUM(o.order_value) AS total_revenue,
           SUM(d.fuel_or_charge_cost) AS total_energy_cost,
           ROUND((SUM(o.order_value) - SUM(d.fuel_or_charge_cost)), 2) AS net_profit,
           ROUND(((SUM(o.order_value) - SUM(d.fuel_or_charge_cost)) / SUM(o.order_value)) * 100, 2) AS margin_percentage
    FROM orders o
    JOIN deliveries d ON o.order_id = d.order_id
    GROUP BY o.service_type
    ORDER BY margin_percentage ASC
")
print(service_profitability)

  service_type total_orders total_revenue total_energy_cost net_profit
1       Retail          224      19444.86           2906.27   16538.59
2      Medical          108       9344.88           1379.48    7965.40
3       Parcel          230      20735.44           3009.01   17726.43
4     Business          126      12279.23           1655.91   10623.32
5    Passenger          262      25463.36           3248.56   22214.80
  margin_percentage
1             85.05
2             85.24
3             85.49
4             86.51
5             87.24


Advanced Analytical Audit: Windows Function

In [15]:
# Objective: Identify the top driver in each hub based on satisfaction ratings
# This demonstrates expert use of SQL Window Functions for localized benchmarking
sqldf("SELECT hub_id, driver_id, avg_rating
       FROM (SELECT hub_id, driver_id, AVG(customer_rating_post_delivery) as avg_rating,
                   RANK() OVER (PARTITION BY hub_id ORDER BY AVG(customer_rating_post_delivery) DESC) as hub_rank
             FROM deliveries
             GROUP BY hub_id, driver_id)
       WHERE hub_rank = 1")

hub_id,driver_id,avg_rating
<chr>,<chr>,<dbl>
H01,D136,5
H01,D126,5
H01,D120,5
H01,D075,5
H01,D025,5
H02,D138,5
H02,D118,5
H02,D099,5
H02,D073,5
